# Tutorial: Business Problem Framing

> **Type:** Tutorial — follow along step by step. Your only job is to make it work.  
> **Time:** ~1.5 hours  
> **Prerequisites:** Module 01, Python basics  
> **What you'll build:** Structured ML problem statements for three realistic SA scenarios, plus a correctly-schemaed dataset ready for modelling.

---

## What We're Building

We'll work through three customer conversations — the kind you actually have as an SA. For each one, we'll do the translation: vague ask → structured problem statement. Then we'll take the best-framed problem and build a synthetic dataset with the right schema: correct label column, sensible features, no data leakage, and a proper train/test split.

You won't train a model today. The point is that **getting here correctly is half the battle**.

## Step 0: Setup

In [ ]:
# !pip install pandas numpy matplotlib seaborn scikit-learn

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

plt.rcParams['figure.figsize'] = (12, 5)
sns.set_style('whitegrid')
print('Setup complete.')

---

## Part 1: Three SA Scenarios

For each scenario, we'll follow the same structure:
1. The customer's actual words
2. The questions you should ask
3. The structured problem statement that results
4. The ML problem type it maps to

### Scenario A: The Telco

**Customer says:**  
*"We're losing customers and we want to use AI to fix it. We have a CRM with transaction history and we think ML could help us predict who's going to leave."*

**Questions to ask:**
- What does "leave" mean exactly — cancel a subscription, go inactive, downgrade?
- Over what time window? Within 30 days? 90 days?
- What happens after you identify an at-risk customer? Is there an intervention team? What's their capacity?
- What data do you actually have, and at what granularity? Per customer per day? Per billing cycle?
- What's the cost of a false positive (reaching out to a customer who wasn't going to churn)?
- What percentage of customers churn in a typical month? (This tells us about class imbalance.)

**Answers received:**
- Churn = account cancellation within 30 days
- Monthly billing data per customer, plus support ticket history
- Intervention team can handle ~2,000 outreach calls per month
- ~5% monthly churn rate across 100,000 customers
- False positive cost: ~$15 per unnecessary call. False negative cost: ~$180 average LTV lost.

**Structured problem statement:**

In [ ]:
scenario_a = {
    "problem_type":         "Binary classification",
    "label":                "churned_within_30_days (1 = yes, 0 = no)",
    "unit_of_analysis":     "One row = one customer at end of billing cycle",
    "prediction_horizon":   "Predict 30 days before potential churn date",
    "features_available":   [
        "months_as_customer",
        "monthly_spend",
        "num_support_tickets_last_90d",
        "days_since_last_login",
        "plan_type",
        "num_products",
        "payment_failures_last_6m",
    ],
    "leakage_risk":         "Do NOT include features computed after churn decision (e.g., cancellation_request_date)",
    "class_imbalance":      "~5% positive class — will require stratified split + class weighting",
    "success_metric":       "Precision ≥ 0.40 at recall ≥ 0.60, targeting top 2,000 customers per month",
    "business_outcome":     "At 15% intervention success rate: ~180 prevented churns/month = ~$32,400 saved/month",
}

for k, v in scenario_a.items():
    print(f"  {k:<25}: {v}")

### Scenario B: The Retailer

**Customer says:**  
*"We want to personalise our website. Right now everyone sees the same homepage. We feel like if we showed people things they were interested in, conversion would go up."*

**Questions to ask:**
- What does "conversion" mean — add to cart? Purchase? Account creation?
- Do you have purchase history per user? How far back?
- How many products are in the catalogue?
- Do you have product metadata (category, price, description)?
- Are most users logged in, or do you have a cold-start problem (many anonymous visitors)?
- Is this real-time (serve a recommendation as the page loads) or batch (email campaign)?

**Answers received:**
- Conversion = purchase within the same session
- 18 months of purchase history, 65% of users are logged in
- 8,000 SKUs with category, price, and description
- Real-time, must respond in < 200ms
- 35% of visits are first-time users with no history (cold-start problem)

**Structured problem statement:**

In [ ]:
scenario_b = {
    "problem_type":         "Ranking (recommendation)",
    "label":                "purchased_in_session (1 = yes, 0 = no), per user-item pair",
    "unit_of_analysis":     "One row = one (user, item) pair in a session",
    "prediction_horizon":   "Real-time, at page load, < 200ms latency budget",
    "features_available":   [
        "user_purchase_history (collaborative signal)",
        "item_category, item_price, item_popularity",
        "session_context: device, time_of_day, referrer",
        "user_demographics (if available)",
    ],
    "cold_start_strategy":  "For 35% anonymous users: fallback to popularity-based ranking by category",
    "latency_constraint":   "Model must serve in < 50ms (leave 150ms for network + rendering)",
    "success_metric":       "Lift in session conversion rate vs. baseline (A/B test, 2 weeks, p < 0.05)",
    "business_outcome":     "Current conversion 2.1%. Target: 2.6%. At 1M sessions/month: +5,000 purchases.",
}

for k, v in scenario_b.items():
    print(f"  {k:<25}: {v}")

### Scenario C: The Manufacturer

**Customer says:**  
*"We want to use AI to predict when our machines are going to break down. Unplanned downtime is costing us millions."*

**Questions to ask:**
- What sensor data do you collect, and at what frequency?
- Do you have historical maintenance records with timestamps?
- What types of failures do you have — are they all the same or different failure modes?
- How much lead time is needed to schedule maintenance?
- What's the cost of a planned maintenance stop vs. an unplanned failure?
- Do all machines have the same failure patterns, or does it vary by machine age/model?

**Answers received:**
- Temperature, vibration, pressure sensors at 1-second intervals for 200 machines
- 3 years of maintenance records; 4 distinct failure modes
- Need 48 hours lead time to order parts and schedule a technician
- Planned maintenance: $2,000. Unplanned failure: $50,000 (lost production + emergency repair)
- Machine age and model vary significantly

**Structured problem statement:**

In [ ]:
scenario_c = {
    "problem_type":         "Multi-class classification (or binary per failure mode)",
    "label":                "failure_within_48h: {0=none, 1=mode_A, 2=mode_B, 3=mode_C, 4=mode_D}",
    "unit_of_analysis":     "One row = one machine at one timestamp (aggregated to hourly windows)",
    "prediction_horizon":   "48-hour horizon — predict failure 48h before it occurs",
    "features_available":   [
        "rolling_mean/std of temp, vibration, pressure (1h, 6h, 24h windows)",
        "hours_since_last_maintenance",
        "machine_age_months",
        "machine_model",
        "operating_load_percent",
    ],
    "leakage_risk":         "Do NOT include sensor readings within the 48h prediction window",
    "class_imbalance":      "Failures rare (~0.5% of hours) — consider anomaly detection approach or oversampling",
    "success_metric":       "Recall ≥ 0.85 for any failure mode; false alarm rate ≤ 5% of predictions",
    "business_outcome":     "At 85% recall on ~50 annual failures: prevent 42 unplanned stops = $2.1M saved/year",
}

for k, v in scenario_c.items():
    print(f"  {k:<25}: {v}")

> **What just happened?** Three "we want AI" asks became three precise ML contracts. Notice how different the resulting models are — ranking, binary classification, multi-class classification — even though all three customers were vaguely asking for "prediction."

---

## Part 2: Building the Dataset Schema

We'll work with Scenario A (churn prediction) for the rest of the tutorial. We'll generate a synthetic dataset that matches the schema defined above and set it up correctly for modelling.

### Step 1: Generate Synthetic Customer Data

In [ ]:
N = 10_000  # 10,000 customers

# Generate features
months_as_customer      = np.random.exponential(scale=24, size=N).astype(int) + 1
monthly_spend           = np.random.lognormal(mean=4.5, sigma=0.6, size=N).round(2)
num_support_tickets     = np.random.poisson(lam=1.2, size=N)
days_since_last_login   = np.random.exponential(scale=8, size=N).astype(int)
plan_type               = np.random.choice(['basic', 'standard', 'premium'], size=N, p=[0.4, 0.4, 0.2])
num_products            = np.random.randint(1, 6, size=N)
payment_failures        = np.random.poisson(lam=0.3, size=N)

# Encode plan type for churn probability calculation
plan_enc = {'basic': 1, 'standard': 0.5, 'premium': 0.1}

# Generate churn label — correlated with risk factors
# Higher support tickets, payment failures, low tenure, long since login → higher churn probability
churn_logit = (
    -3.5
    + 0.08 * num_support_tickets
    + 0.12 * payment_failures
    + 0.03 * days_since_last_login
    - 0.02 * np.log1p(months_as_customer)
    + np.array([plan_enc[p] for p in plan_type])
    - 0.1  * num_products
    + np.random.normal(0, 0.5, N)  # noise
)
churn_prob = 1 / (1 + np.exp(-churn_logit))
churned = (np.random.uniform(size=N) < churn_prob).astype(int)

df = pd.DataFrame({
    'customer_id':              np.arange(1, N + 1),
    'months_as_customer':       months_as_customer,
    'monthly_spend':            monthly_spend,
    'num_support_tickets_90d':  num_support_tickets,
    'days_since_last_login':    days_since_last_login,
    'plan_type':                plan_type,
    'num_products':             num_products,
    'payment_failures_6m':      payment_failures,
    'churned_within_30d':       churned,   # <-- this is our label
})

print(f"Dataset shape: {df.shape}")
print(f"Churn rate: {df['churned_within_30d'].mean():.1%}")
df.head()

### Step 2: Inspect the Label Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Label distribution
label_counts = df['churned_within_30d'].value_counts()
axes[0].bar(['Did not churn (0)', 'Churned (1)'], label_counts.values, 
            color=['#4E79A7', '#E15759'])
axes[0].set_title('Label Distribution')
axes[0].set_ylabel('Count')
for i, v in enumerate(label_counts.values):
    axes[0].text(i, v + 50, f'{v:,}\n({v/N:.1%})', ha='center', fontsize=10)

# Churn rate by plan type
churn_by_plan = df.groupby('plan_type')['churned_within_30d'].mean().sort_values(ascending=False)
axes[1].bar(churn_by_plan.index, churn_by_plan.values, color=['#E15759', '#F28E2B', '#59A14F'])
axes[1].set_title('Churn Rate by Plan Type')
axes[1].set_ylabel('Churn Rate')
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))

plt.tight_layout()
plt.savefig('churn_label_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

> **What to notice:** The dataset is imbalanced — far more non-churners than churners. This is real: most customers don't churn in any given month. When we get to modelling (Module 11), we'll handle this with stratified splits and class weighting. For now, just notice it exists.

### Step 3: Feature-Label Relationships

Before any modelling, always plot how features relate to the label. This is EDA (Module 09) in miniature.

In [ ]:
numeric_features = ['months_as_customer', 'num_support_tickets_90d', 
                    'days_since_last_login', 'payment_failures_6m']

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for ax, feat in zip(axes, numeric_features):
    churned     = df[df['churned_within_30d'] == 1][feat]
    not_churned = df[df['churned_within_30d'] == 0][feat]
    
    ax.hist(not_churned, bins=30, alpha=0.6, label='No churn', color='#4E79A7', density=True)
    ax.hist(churned,     bins=30, alpha=0.6, label='Churned',  color='#E15759', density=True)
    ax.set_title(feat.replace('_', '\n'), fontsize=9)
    ax.set_ylabel('Density')
    ax.legend(fontsize=8)

plt.suptitle('Feature Distributions: Churned vs Not Churned', fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

### Step 4: Train / Test Split

We always hold out a test set before any modelling. The rule: **the model is never allowed to see the test set until final evaluation**. This simulates production — you build on past data, evaluate on future data.

In [ ]:
# Separate features (X) from label (y)
# Drop non-predictive columns: customer_id is an identifier, not a feature
feature_cols = [
    'months_as_customer', 'monthly_spend', 'num_support_tickets_90d',
    'days_since_last_login', 'num_products', 'payment_failures_6m',
    # plan_type needs encoding first — we'll handle this in Module 14 (Feature Engineering)
]
label_col = 'churned_within_30d'

X = df[feature_cols]
y = df[label_col]

# stratify=y ensures both splits have the same churn rate
# This is critical with imbalanced data
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_SEED,
    stratify=y       # <-- preserves class balance in both splits
)

print(f"Training set:   {X_train.shape[0]:,} rows | churn rate: {y_train.mean():.1%}")
print(f"Test set:       {X_test.shape[0]:,} rows  | churn rate: {y_test.mean():.1%}")
print(f"\nFeatures ready for modelling: {list(X_train.columns)}")

> **What just happened?** You now have a correctly-schemaed dataset — label separated from features, no leakage, stratified split preserving class balance. This is the scaffold every subsequent model will be built on.

### Step 5: Save the Dataset

In [ ]:
# Save the full dataset for use in later modules
df.to_csv('churn_dataset.csv', index=False)
print(f"Saved churn_dataset.csv ({len(df):,} rows)")

# Quick summary
print("\nDataset summary:")
print(df.describe().round(2))

---

## What You Built

You translated three realistic SA customer asks into precise ML problem statements, and built a correctly-schemaed synthetic churn dataset: label defined, features identified, leakage risks flagged, and a stratified train/test split in place.

## What's Next

- Read `explanation.md` if you haven't — the precision/recall trade-off section is especially important for the business metrics we defined.
- Open `howto.md` for the problem statement template you can use in real customer conversations.
- **Extension challenge:** Write a problem statement for a fourth scenario: a bank wants to "use AI for credit decisions."

**Next module:** `03_math_stats_probability_linear_algebra` — the mathematical foundations that underpin every model you'll build.

---

**AWS SageMaker note:** The synthetic dataset we generated here could be stored in S3 and registered as a SageMaker Feature Store feature group. In Module 26 (Cloud ML Platforms) we'll do exactly that.